# ProGen3 Protein Sequence Generation and Scoring

This notebook demonstrates both functions of the ProGen3 tool. In the first section, we use `run_progen3_sample` to generate protein sequences using forward, reverse, and unconditional generation modes. In the second section, we use `run_progen3_score` to compute bidirectional log-likelihood and perplexity metrics that measure how natural a sequence appears to the model.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("progen3")
display_overview("progen3")
display_docs_section("progen3", "Background")

# ProGen3

> ProGen3 is a Mixture-of-Experts (MoE) protein language model from Profluent, based on the Mixtral/Mistral architecture. It supports autoregressive protein sequence generation in both forward (N→C) and reverse (C→N) directions, as well as bidirectional sequence scoring that averages forward and reverse log-likelihoods for more robust evaluation.

## Background

[Protein language models](https://en.wikipedia.org/wiki/Protein_language_model) learn the statistical patterns of natural protein sequences, capturing evolutionary constraints on amino acid usage. ProGen3's bidirectional scoring (averaging N→C and C→N likelihoods) provides a more robust assessment of sequence naturalness than unidirectional models, since protein function depends on the complete [3D structure](https://en.wikipedia.org/wiki/Protein_structure) rather than just the N-to-C reading frame.

The [Mixture-of-Experts](https://en.wikipedia.org/wiki/Mixture_of_experts) architecture activates only a subset of parameters per token, allowing larger model capacity without proportionally increasing compute cost.

## Available tools

In [2]:
display_available_tools("progen3")

- **`run_progen3_sample()`** — Sample protein sequences using ProGen3 language model
- **`run_progen3_score()`** — Score protein sequences using ProGen3 language model

### `run_progen3_sample`

ProGen3 is a Mixture-of-Experts protein language model that supports both forward (N-to-C) and reverse (C-to-N) autoregressive generation. Forward mode extends a sequence from an N-terminal prompt, while reverse mode extends from a C-terminal prompt — useful when a conserved C-terminal motif or membrane anchor must be preserved. Passing an empty string as the prompt produces unconditional generation, where the model samples an entire sequence from scratch. Available checkpoints range from 112M to 3B parameters.

In [3]:
from proto_tools import (
    ProGen3SampleInput,
    ProGen3SampleConfig,
    run_progen3_sample,
)

In [4]:
# Display docs
display_api_reference("progen3", "input", "run_progen3_sample")

# Forward generation from an N-terminal prompt
inputs = ProGen3SampleInput(prompts=["MKTLVIVTGA"])

**Input** — `ProGen3SampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `List[string]` | required | Amino acid prompt sequences for protein generation. Use `direction` in `ProGen3SampleConfig` to control generation direction. Pass an empty string (`""`) for unconditional generation. Can be a single string or a list of strings. |

In [5]:
# Display config docs
display_api_reference("progen3", "config", "run_progen3_sample")

# Configure forward sampling | see docs above for all fields
config = ProGen3SampleConfig(
    model_checkpoint="progen3-762m",
    direction="forward",
    temperature=0.2,
    top_p=0.95,
    max_new_tokens=100,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen3SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `progen3-762m` | ProGen3 model checkpoint to use. Options: Available options: `progen3-112m`, `progen3-219m`, `progen3-339m`, `progen3-762m`, `progen3-1b`, `progen3-3b` |
| `direction` | `enum` | `forward` | Generation direction. `"forward"` generates N→C (left to right), `"reverse"` generates C→N (right to left). Default: `"forward"`. Available options: `forward`, `reverse` |
| `temperature` | `number` | `0.2` | Scales the randomness of sampling: |
| `top_p` | `number` | `0.95` | Nucleus sampling parameter. Smallest set of tokens whose cumulative probability mass is at least `top_p`. |
| `max_new_tokens` | `integer` | `256` | Maximum number of new tokens to generate per prompt. Does not include prompt tokens. Default: 256. |
| `min_new_tokens` | `integer` | `1` | Minimum number of new tokens to generate per prompt. Generation will not stop before this many tokens. Default: 1. |
| `num_sequences` | `integer` | `1` | Number of sequences to generate per prompt. |
| `include_prompt_in_output` | `boolean` | `True` | Whether to include the input prompt residues in the output sequence. If `True` (default), returned sequences include both the prompt and newly generated residues. If `False`, only newly generated residues are returned. |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously on GPU. |
| `local_path` | `string` |  | Optional path to local model weights directory. If provided, loads model from local filesystem instead of downloading from HuggingFace. Default: `None`. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
result = run_progen3_sample(inputs, config)

Running run_progen3_sample [00:00]

In [7]:
# Display output docs
display_api_reference("progen3", "output", "run_progen3_sample")

# Show the generated sequence
print(f"Generated {len(result.sequences)} sequence(s)")
print(f"Prompt:   {inputs.prompts[0]}")
print(f"Sequence: {result.sequences[0][:80]}...")

**Output** — `ProGen3SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Generated protein sequences. Each sequence contains amino acid characters. Special tokens and direction indicators are always stripped. If `include_prompt_in_output=True` (default), sequences include the input prompt residues; if `False`, only newly generated residues are returned. |

Generated 1 sequence(s)
Prompt:   MKTLVIVTGA
Sequence: MKTLVIVTGASSGIGAATARLLAARGHPLLLVARRLDRLEALAAELGAAHAVAADLTDPAAVAAAVAAVEARHGPVDVLV...


### Reverse Generation

Setting `direction="reverse"` causes the model to extend a C-terminal prompt toward the N-terminus. This is useful when the design constraint is at the end of the protein, such as a purification tag, a membrane anchor, or a conserved C-terminal domain boundary.

In [8]:
# Generate from a C-terminal prompt in reverse direction
inputs_rev = ProGen3SampleInput(prompts=["RYTEAFLK"])
config_rev = ProGen3SampleConfig(
    model_checkpoint="progen3-762m",
    direction="reverse",
    max_new_tokens=30,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [9]:
result_rev = run_progen3_sample(inputs_rev, config_rev)
print(f"Prompt:" + " "*len(result_rev.sequences[0]) + "<-" + inputs_rev.prompts[0])
print(f"Generated:       {result_rev.sequences[0]}")

Running run_progen3_sample [00:00]

Prompt:                                        <-RYTEAFLK
Generated:       VKEADLPVLAANALKDACGLTNPRPATPEEVARYTEAFLK


### Unconditional Generation

Passing an empty string as the prompt produces unconditional generation, where the model samples an entire protein sequence from scratch without any seed context. This is useful for exploring the space of plausible proteins or for generating diverse starting points that can be refined with downstream tools.

In [10]:
# Unconditional generation with an empty prompt
inputs_uncond = ProGen3SampleInput(prompts=[""])
config_uncond = ProGen3SampleConfig(
    max_new_tokens=200,
    device="cuda",  # Change to "cpu" if no GPU available
)

result_uncond = run_progen3_sample(inputs_uncond, config_uncond)
print(f"Generated: {result_uncond.sequences[0][:80]}...")
print(f"Length: {len(result_uncond.sequences[0])} residues")

Running run_progen3_sample [00:00]

Generated: MALLLLAACGGGSSSSSGGSGGGGGSGGGGGSGGGGGSGGGGGSGGGGGSGGGGGSGGGGGSGGGGGSGGGGGSGGGGGS...
Length: 202 residues


### `run_progen3_score`

ProGen3 scoring computes bidirectional log-likelihoods by averaging the forward (N-to-C) and reverse (C-to-N) autoregressive probabilities at each position. This bidirectional approach produces a more robust naturalness estimate than unidirectional models. The resulting perplexity measures how consistent a sequence is with the patterns learned from natural proteins, making it useful for ranking design candidates, comparing wild-type and mutant variants, or filtering sequences before expensive experimental validation.

In [11]:
from proto_tools import (
    ProGen3ScoringInput,
    ProGen3ScoringConfig,
    run_progen3_score,
)

In [12]:
# Display docs
display_api_reference("progen3", "input", "run_progen3_score")

# Score a hemoglobin alpha fragment and a signal peptide
score_inputs = ProGen3ScoringInput(
    sequences=[
        "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF",  # Hemoglobin alpha fragment
        "MKTLLILAVVAAALA",  # Signal peptide
    ]
)

**Input** — `ProGen3ScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequences to score. |

In [13]:
# Display config docs
display_api_reference("progen3", "config", "run_progen3_score")

# Configure scoring | see docs above for all fields
score_config = ProGen3ScoringConfig(
    model_checkpoint="progen3-762m",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen3ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `progen3-762m` | ProGen3 model checkpoint to use. Options include `"progen3-112m"` (112M), `"progen3-219m"` (219M), `"progen3-339m"` (339M), `"progen3-762m"` (762M), `"progen3-1b"` (1B), `"progen3-3b"` (3B). Available options: `progen3-112m`, `progen3-219m`, `progen3-339m`, `progen3-762m`, `progen3-1b`, `progen3-3b` |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously on GPU. |
| `reduction` | `enum` | `mean` | How to aggregate per-token log-likelihoods. `"mean"` averages over tokens, `"sum"` sums them. Available options: `mean`, `sum` |
| `local_path` | `string` |  | Optional path to local model weights directory. If provided, loads model from local filesystem instead of downloading from HuggingFace. Default: `None`. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [14]:
# Run the scoring function
score_result = run_progen3_score(score_inputs, score_config)

Running run_progen3_score [00:00]

In [15]:
# Display output docs
display_api_reference("progen3", "output", "run_progen3_score")

# Show scoring results for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {score_inputs.sequences[i][:40]}...")
    print(f"    Log-likelihood:     {score.metrics['log_likelihood']:.4f}")
    print(f"    Avg log-likelihood: {score.metrics['avg_log_likelihood']:.4f}")
    print(f"    Perplexity:         {score.metrics['perplexity']:.4f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[SequenceScores]` | required | List of scoring outputs, one per input sequence. Each entry contains metrics (log\_likelihood, avg\_log\_likelihood, perplexity) and optional per-position logits. |
| `metrics` | `Dict[string, number]` |  | Dictionary of scalar scoring metrics. |
| `logits` | `array` |  | Optional per-position logits array. |
| `vocab` | `array` |  | Optional token ordering for logits; logits\[:, j] corresponds to vocab\[j]. |
| `per_position_metrics` | `Dict[string, List[number]]` |  | Optional per-position scoring metrics, keyed by metric name. Each value is a list of length equal to the input sequence, with `None` at positions where that metric is not available. |

Sequence 1: MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTT...
    Log-likelihood:     -2.6656
    Avg log-likelihood: -2.6656
    Perplexity:         14.3762
Sequence 2: MKTLLILAVVAAALA...
    Log-likelihood:     -2.2828
    Avg log-likelihood: -2.2828
    Perplexity:         9.8044


In [16]:
for i, score in enumerate(score_result.scores):
    # Per-position metrics: forward, reverse, and bidirectional log-likelihoods
    if score.per_position_metrics:
        per_pos = score.per_position_metrics
        seq = score_inputs.sequences[i]
        print(f"\n    Per-position log-likelihoods (first 10 residues):")
        print(f"    {'Pos':>4} {'AA':>3} {'Forward':>9} {'Reverse':>9} {'Bidir':>9}")
        for j in range(min(10, len(seq))):
            fwd = per_pos["forward_log_likelihood"][j]
            rev = per_pos["reverse_log_likelihood"][j]
            bid = per_pos["log_likelihood"][j]
            fwd_str = f"{fwd:.4f}" if fwd is not None else "N/A"
            rev_str = f"{rev:.4f}" if rev is not None else "N/A"
            bid_str = f"{bid:.4f}" if bid is not None else "N/A"
            print(f"    {j + 1:>4} {seq[j]:>3} {fwd_str:>9} {rev_str:>9} {bid_str:>9}")
    print()


    Per-position log-likelihoods (first 10 residues):
     Pos  AA   Forward   Reverse     Bidir
       1   M       N/A   -1.1016   -1.1016
       2   V   -0.7249   -2.5563   -1.6406
       3   L   -0.0041   -2.8325   -1.4183
       4   S   -3.4031   -2.3787   -2.8909
       5   P   -2.4732   -3.0746   -2.7739
       6   A   -2.5107   -2.7776   -2.6441
       7   D   -3.0957   -3.2145   -3.1551
       8   K   -2.3673   -3.5445   -2.9559
       9   T   -2.9745   -2.5685   -2.7715
      10   N   -3.2092   -3.0480   -3.1286


    Per-position log-likelihoods (first 10 residues):
     Pos  AA   Forward   Reverse     Bidir
       1   M       N/A   -2.9280   -2.9280
       2   K   -0.7249   -1.8295   -1.2772
       3   T   -0.0041   -2.2126   -1.1083
       4   L   -2.0789   -3.3057   -2.6923
       5   L   -2.6061   -2.6980   -2.6520
       6   I   -2.0201   -1.5807   -1.8004
       7   L   -1.9014   -2.7948   -2.3481
       8   A   -2.0938   -2.1050   -2.0994
       9   V   -1.9270   -1.9

## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for downstream use. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [17]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="progen3_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'progen3_sequences.fasta'}")

score_result.export(name="progen3_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'progen3_scores.csv'}")

Generated sequences exported to example_output/progen3_sequences.fasta
Scores exported to example_output/progen3_scores.csv
